In [13]:
import numpy as np
import pandas as pd
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsOneClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.utils import resample

df = pd.read_csv('winequality-red-clean.csv', delimiter=',')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1458 entries, 0 to 1457
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1458 non-null   float64
 1   volatile acidity      1458 non-null   float64
 2   citric acid           1458 non-null   float64
 3   residual sugar        1458 non-null   float64
 4   chlorides             1458 non-null   float64
 5   free sulfur dioxide   1458 non-null   float64
 6   total sulfur dioxide  1458 non-null   float64
 7   density               1458 non-null   float64
 8   pH                    1458 non-null   float64
 9   sulphates             1458 non-null   float64
 10  alcohol               1458 non-null   float64
 11  quality               1458 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 136.8 KB


In [14]:

'''
Selected Features classification:
 Index(['volatile acidity', 'citric acid', 'total sulfur dioxide', 'sulphates',
       'alcohol'], dtype='object')
'''

selected_features = ['volatile acidity', 'citric acid', 'total sulfur dioxide', 'sulphates',
                     'alcohol']
X_selected = df[selected_features]
y = df['quality']

standard_scaler = StandardScaler()
X_selected = standard_scaler.fit_transform(X_selected)

X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)


In [15]:
def train_test_accuracy(train_accuracy, test_accuracy):
    print(f'Training Accuracy: {(train_accuracy * 100).__round__(3)}%')
    print(f'Testing Accuracy: {(test_accuracy * 100).__round__(3)}%')


In [16]:
def calculate_bias_variance(model, X, y, num_samples=100):
    all_predictions = np.zeros((num_samples, X.shape[0]))

    for i in range(num_samples):
        X_resampled, y_resampled = resample(X, y)

        model.fit(X_resampled, y_resampled)

        all_predictions[i, :] = model.predict(X)

    mean_predictions = np.mean(all_predictions, axis=0)

    bias_squared = np.mean((mean_predictions - y) ** 2)

    variance = np.mean(np.var(all_predictions, axis=0))

    print(f"Bias^2: {bias_squared}")
    print(f"Variance: {variance}")


In [17]:
from sklearn.multiclass import OneVsRestClassifier

# KNN & SVM Models
knn_1 = KNeighborsClassifier(weights='distance', leaf_size=5)

knn_2 = KNeighborsClassifier(weights='distance', leaf_size=10, n_neighbors=5)

knn_3 = KNeighborsClassifier(weights='uniform', leaf_size=15, n_neighbors=7)

ovr_linear_model = OneVsRestClassifier(SVC(kernel='linear', C=1000))  # Best Parameters: {'estimator__C': 1000}

ovr_rbf_model = OneVsRestClassifier(
    SVC(kernel='rbf', C=1, gamma=0.1))  #Best Parameters: {'estimator__C': 1, 'estimator__gamma': 0.1}

ovo_linear_model = OneVsOneClassifier(SVC(kernel='linear', C=10))  #Best Parameters: {'estimator__C': 10}

ovo_rbf_model = OneVsOneClassifier(
    SVC(kernel='rbf', C=1, gamma=0.1))  # Best Parameters: {'estimator__C': 1, 'estimator__gamma': 0.1}


In [18]:
voting_clf = VotingClassifier(estimators=[
    ('knn_1', knn_1),
    ('knn_2', knn_2),
    ('knn_3', knn_3),
    ('ovr_linear_model', ovr_linear_model),
    ('ovr_rbf_model', ovr_rbf_model),
    ('ovo_linear_model', ovo_linear_model),
    ('ovo_rbf_model', ovo_rbf_model)
], voting='hard')
# Train the ensemble model
voting_clf.fit(X_train, y_train)

# Make predictions on the test set
y_pred = voting_clf.predict(X_test)

# Calculate the accuracy of the ensemble
accuracy = accuracy_score(y_test, y_pred)
print(f"Ensemble Voting Classifier Accuracy: {accuracy * 100:.2f}%")
#calculate_bias_variance(voting_clf, X_test, y_test)

Ensemble Voting Classifier Accuracy: 63.36%


In [ ]:
from sklearn.ensemble import BaggingClassifier, VotingClassifier

# Wrap each estimator with BaggingClassifier, reduce n_estimators to 3 for faster training
bagged_knn_1 = BaggingClassifier(knn_1, n_estimators=9, random_state=42, n_jobs=-1)
bagged_knn_2 = BaggingClassifier(knn_2, n_estimators=9, random_state=42, n_jobs=-1)
bagged_knn_3 = BaggingClassifier(knn_3, n_estimators=9, random_state=42, n_jobs=-1)
bagged_ovr_linear = BaggingClassifier(ovr_linear_model, n_estimators=5, random_state=42, n_jobs=-1)
bagged_ovr_rbf = BaggingClassifier(ovr_rbf_model, n_estimators=9, random_state=42, n_jobs=-1)
bagged_ovo_linear = BaggingClassifier(ovo_linear_model, n_estimators=9, random_state=42, n_jobs=-1)
bagged_ovo_rbf = BaggingClassifier(ovo_rbf_model, n_estimators=9, random_state=42, n_jobs=-1)

# Combine the bagged models in the VotingClassifier
voting_clf_bagged = VotingClassifier(estimators=[
    ('bagged_knn_1', bagged_knn_1),
    ('bagged_knn_2', bagged_knn_2),
    ('bagged_knn_3', bagged_knn_3),
    ('bagged_ovr_linear', bagged_ovr_linear),
    ('bagged_ovr_rbf', bagged_ovr_rbf),
    ('bagged_ovo_linear', bagged_ovo_linear),
    ('bagged_ovo_rbf', bagged_ovo_rbf)
], voting='hard', n_jobs=-1)

# Train the bagged VotingClassifier
voting_clf_bagged.fit(X_train, y_train)

# Make predictions and calculate accuracy
y_pred_bagged = voting_clf_bagged.predict(X_test)
accuracy_bagged = accuracy_score(y_test, y_pred_bagged)
print(f"Accuracy (Bagged Voting Classifier with 3 estimators): {accuracy_bagged * 100:.2f}%")
